In [3]:
import math

def resolucao_ec3_final_corrigida():
    print("=======================================================")
    print("   VERIFICAÇÃO DE SEGURANÇA (EC3) - VERSÃO FINAL")
    print("=======================================================")
    print("Nota: O programa converte tudo para unidades SI (m, N, Pa)")
    print("      antes de calcular. Insira nas unidades pedidas.\n")

    # ==========================================
    # 1. SELEÇÃO E MATERIAIS
    # ==========================================
    print("Tipo de Elemento:")
    print("1: Pilar (Column)")
    print("2: Viga de Cobertura (Roof Beam)")
    try:
        tipo = int(input("Escolha (1 ou 2): "))
    except ValueError: return

    # --- MATERIAIS (Input GPa/MPa -> SI: Pa) ---
    try:
        E_gpa = float(input("\nMódulo de Young (E) [GPa]: "))
        E = E_gpa * 1.0e9  # Pascal
        
        fy_mppa = float(input("Tensão de Cedência (fy) [MPa]: "))
        fy = fy_mppa * 1.0e6 # Pascal
    except ValueError: return

    # Cálculo Lambda_1 (Exato via Fórmula de Euler)
    lambda_1 = math.pi * math.sqrt(E / fy)
    print(f">> Lambda_1 calculado: {round(lambda_1, 4)}")

    # ==========================================
    # 2. ENCURVADURA POR FLEXÃO (EIXO Y-Y)
    # ==========================================
    print("\n--- 2.1 ENCURVADURA FLEXÃO EIXO Y-Y (FORTE) ---")
    
    try:
        if tipo == 1: # Pilar
            beta = float(input("Coef. correção comprimento (beta): "))
            Ly_m = float(input("Comprimento real (Ly) [m]: "))
            Le_y = beta * Ly_m
        else: # Viga
            Le_y_m = float(input("Comprimento encurvadura (Le,y) [m]: "))
            Le_y = Le_y_m
        
        iy_mm = float(input("Raio de giração (iy) [mm]: "))
        iy = iy_mm / 1000.0 # Converter para METROS
    except ValueError: return

    # Cálculos SI
    lambda_y = Le_y / iy
    lambda_bar_y = lambda_y / lambda_1

    print(f"\n>> Resultados Y-Y:")
    print(f"   Le,y          = {Le_y:.3f} m")
    print(f"   Lambda_y      = {round(lambda_y, 4)}")
    print(f"   Lambda_bar_y  = {round(lambda_bar_y, 4)}")

    # --- INPUT GEOMETRIA PARA SELEÇÃO DE CURVA ---
    print("\n[Dados para seleção da Curva de Encurvadura]")
    try:
        h_mm = float(input("Altura total (h) [mm]: "))
        b_mm = float(input("Largura (b) [mm]: "))
        tf_mm = float(input("Espessura banzo (tf) [mm]: "))
        
        # Conversão imediata para SI (metros) para uso posterior
        h = h_mm / 1000.0
        b = b_mm / 1000.0
        tf = tf_mm / 1000.0
        
        # Auxílio Visual
        ratio_hb = h_mm / b_mm
        print(f"--> Razão h/b = {ratio_hb:.2f}")
        if tf_mm <= 40: print(f"--> tf = {tf_mm} mm (<= 40 mm)")
        elif tf_mm <= 100: print(f"--> tf = {tf_mm} mm (40 < tf <= 100)")
        else: print(f"--> tf = {tf_mm} mm (> 100 mm)")
        
    except ValueError: return

    try:
        alpha_y = float(input("\nFator imperfeição (alpha_y) [a=0.21, b=0.34...]: "))
    except ValueError: return

    # Cálculo Chi Y
    phi_y = 0.5 * (1 + alpha_y * (lambda_bar_y - 0.2) + lambda_bar_y**2)
    delta_y = phi_y**2 - lambda_bar_y**2
    if delta_y < 0: delta_y = 0
    
    chi_y = 1 / (phi_y + math.sqrt(delta_y))
    if chi_y > 1.0: chi_y = 1.0

    print(f">> Phi_y         = {round(phi_y, 4)}")
    print(f">> Chi_y (Xy)    = {round(chi_y, 4)}")

    # ==========================================
    # 3. ENCURVADURA POR FLEXÃO (EIXO Z-Z)
    # ==========================================
    print("\n--- 2.2 ENCURVADURA FLEXÃO EIXO Z-Z (FRACO) ---")
    calc_zz = input("Calcular encurvadura Z-Z? (s/n): ").lower()
    
    chi_z = None 
    lambda_bar_z = None

    if calc_zz == 's':
        try:
            Le_z_m = float(input("Comprimento encurvadura (Le,z) [m]: "))
            Le_z = Le_z_m
            
            iz_mm = float(input("Raio de giração (iz) [mm]: "))
            iz = iz_mm / 1000.0 # Converter para METROS
            
            # Cálculos SI
            lambda_z = Le_z / iz
            lambda_bar_z = lambda_z / lambda_1

            print(f"\n>> Resultados Z-Z:")
            print(f"   Lambda_z      = {round(lambda_z, 4)}")
            print(f"   Lambda_bar_z  = {round(lambda_bar_z, 4)}")

            alpha_z = float(input("Fator imperfeição (alpha_z) [b=0.34, c=0.49...]: "))
            
            # Cálculo Chi Z
            phi_z = 0.5 * (1 + alpha_z * (lambda_bar_z - 0.2) + lambda_bar_z**2)
            delta_z = phi_z**2 - lambda_bar_z**2
            if delta_z < 0: delta_z = 0
            
            chi_z = 1 / (phi_z + math.sqrt(delta_z))
            if chi_z > 1.0: chi_z = 1.0

            print(f">> Phi_z         = {round(phi_z, 4)}")
            print(f">> Chi_z (Xz)    = {round(chi_z, 4)}")
        except ValueError: return
    else:
        print(">> Passo 2.2 ignorado.")

    # ==========================================
    # 4. ENCURVADURA LATERAL-TORSIONAL (LT)
    # ==========================================
    print("\n--- 2.3 ENCURVADURA LATERAL (TORSÃO) ---")
    print("Insira propriedades em falta:")
    try:
        # h, b, tf já foram pedidos. Falta tw, Iz, Wpl
        tw = float(input("Espessura alma (tw) [mm]: ")) / 1000.0
        
        Iz_mm4 = float(input("Inércia Iz [mm^4]: "))
        Iz = Iz_mm4 * 1.0e-12 
        
        Wply_mm3 = float(input("Módulo Plástico Wpl,y [mm^3]: "))
        Wply = Wply_mm3 * 1.0e-9
    except ValueError: return

    # Propriedades Torsionais (SI)
    hm = h - tf
    Iw = (Iz * (hm**2)) / 4.0
    It = (1/3.0) * (2 * b * tf**3 + (h - 2*tf) * tw**3)

    print(f"\n>> Propriedades SI:")
    print(f"   hm = {hm:.4f} m")
    print(f"   Iw = {Iw:.4e} m^6")
    print(f"   It = {It:.4e} m^4")

    print("\nCálculo de Mcr:")
    try:
        C1 = float(input("Coeficiente C1 (Tabela): "))
        L_cr_LT_m = float(input("Comprimento Encurvadura LT (Lcr,LT) [m]: "))
        L_cr_LT = L_cr_LT_m
        
        G_gpa_in = input("Módulo Transversal (G) [GPa] (Enter p/ 81): ")
        if G_gpa_in == "": G = 81.0 * 1.0e9
        else: G = float(G_gpa_in) * 1.0e9
    except ValueError: return

    # Cálculo Mcr (Tudo em SI) -> Resultado em Nm
    pi2_E_Iz = (math.pi**2) * E * Iz
    L2 = L_cr_LT**2
    
    term1 = (math.pi**2 * E * Iz) / L2
    term2 = Iw / Iz
    term3 = (L2 * G * It) / (math.pi**2 * E * Iz)
    
    Mcr_Nm = C1 * term1 * math.sqrt(term2 + term3)
    
    # Lambda LT
    lambda_bar_LT = math.sqrt((Wply * fy) / Mcr_Nm)

    print(f"\n>> Resultados Mcr:")
    print(f"   Mcr           = {round(Mcr_Nm / 1000.0, 2)} kNm")
    print(f"   Lambda_bar_LT = {round(lambda_bar_LT, 4)}")

    try:
        alpha_LT = float(input("Fator imperfeição (alpha_LT) [Se Laminado h/b>2 -> b(0.34)]: "))
    except ValueError: return

    # Cálculo Chi LT
    phi_LT = 0.5 * (1 + alpha_LT * (lambda_bar_LT - 0.2) + lambda_bar_LT**2)
    delta_LT = phi_LT**2 - lambda_bar_LT**2
    if delta_LT < 0: delta_LT = 0
    
    chi_LT = 1 / (phi_LT + math.sqrt(delta_LT))
    if chi_LT > 1.0: chi_LT = 1.0

    print(f">> Phi_LT        = {round(phi_LT, 4)}")
    print(f">> Chi_LT        = {round(chi_LT, 4)}")

    # ==========================================
    # 5. FATORES DE INTERAÇÃO (Kyy, Kzy)
    # ==========================================
    print("\n" + "="*40)
    cont_kyy = input("Deseja calcular k_yy? (s/n): ").lower()
    
    if cont_kyy == 's':
        print("\n--- 3.1 CÁLCULO DE k_yy (Método 2) ---")
        try:
            # Inputs Força (kN -> SI: N)
            N_Ed_kn = float(input("Esforço Normal N_Ed [kN]: "))
            N_Ed = N_Ed_kn * 1000.0
            
            Area_mm2 = float(input("Área da Secção [mm^2]: "))
            Area = Area_mm2 * 1.0e-6 
            
            gamma_m1 = float(input("Gamma_M1 (normalmente 1.0): "))
            Cmy = float(input("Fator Cmy (Tabela B.3): "))
        except ValueError: return

        # Capacidade Axial (N)
        NRk = Area * fy 
        
        # Termo N (CORRIGIDO: (Ned / (chi * NRk)) / Gamma_M1)
        # Atenção: Se Gamma=1.0, o resultado é o mesmo, mas matematicamente segue o pedido.
        n_ratio = N_Ed / ((chi_y * NRk) / gamma_m1)
        
        # kyy = Cmy * (1 + (lambda_bar_y - 0.2) * n)
        kyy_calc = Cmy * (1 + (lambda_bar_y - 0.2) * n_ratio)
        
        # Limite: Cmy * (1 + 0.8 * n)
        kyy_limit = Cmy * (1 + 0.8 * n_ratio)
        
        kyy = min(kyy_calc, kyy_limit)
        
        print(f"\n>> Resultado Interação k_yy:")
        print(f"   Termo n ratio = {round(n_ratio, 4)}")
        print(f"   Comparação    : {round(kyy_calc, 4)} <= {round(kyy_limit, 4)}")
        print(f"   k_yy FINAL    = {round(kyy, 4)}")

        # Só pergunta por kzy se o chi_z foi calculado
        if chi_z is not None:
            cont_kzy = input("\nDeseja calcular k_zy? (s/n): ").lower()
            if cont_kzy == 's':
                print("\n--- 3.2 CÁLCULO DE k_zy (Método 2) ---")
                try:
                    CmLT = float(input("Fator CmLT: "))
                except ValueError: return
                
                # Termo N com Chi_z (CORRIGIDO)
                n_ratio_z = N_Ed / ( (chi_z * NRk) / gamma_m1)
                
                # kzy formula
                denom = CmLT - 0.25
                if denom == 0: denom = 0.0001
                
                kzy_calc = 1 - (0.1 * lambda_bar_z / denom) * n_ratio_z
                kzy_limit = 1 - (0.1 / denom) * n_ratio_z
                
                kzy = max(kzy_calc, kzy_limit)
                
                print(f"\n>> Resultado Interação k_zy:")
                print(f"   Termo n_z ratio = {round(n_ratio_z, 4)}")
                print(f"   Comparação      : {round(kzy_calc, 4)} >= {round(kzy_limit, 4)}")
                print(f"   k_zy FINAL      = {round(kzy, 4)}")
        else:
            print("\n(Cálculo de k_zy indisponível pois encurvadura Z-Z foi ignorada)")

    print("\n=======================================================")
    print("   FIM DA RESOLUÇÃO")
    print("=======================================================")

# Executar
resolucao_ec3_final_corrigida()

   VERIFICAÇÃO DE SEGURANÇA (EC3) - VERSÃO FINAL
Nota: O programa converte tudo para unidades SI (m, N, Pa)
      antes de calcular. Insira nas unidades pedidas.

Tipo de Elemento:
1: Pilar (Column)
2: Viga de Cobertura (Roof Beam)


Escolha (1 ou 2):  2

Módulo de Young (E) [GPa]:  210
Tensão de Cedência (fy) [MPa]:  275


>> Lambda_1 calculado: 86.8147

--- 2.1 ENCURVADURA FLEXÃO EIXO Y-Y (FORTE) ---


Comprimento encurvadura (Le,y) [m]:  15.529
Raio de giração (iy) [mm]:  243



>> Resultados Y-Y:
   Le,y          = 15.529 m
   Lambda_y      = 63.9053
   Lambda_bar_y  = 0.7361

[Dados para seleção da Curva de Encurvadura]


Altura total (h) [mm]:  600
Largura (b) [mm]:  220
Espessura banzo (tf) [mm]:  19


... (10 outputs left out)

>> Phi_LT        = 1.0477
>> Chi_LT        = 0.6471



Deseja calcular k_yy? (s/n):  s



--- 3.1 CÁLCULO DE k_yy (Método 2) ---


Esforço Normal N_Ed [kN]:  59.957
Área da Secção [mm^2]:  156000
Gamma_M1 (normalmente 1.0):  
